# 🌊 Week 4, Day 1 — June 8, 2026
## Extract & Map High-Density Plastic Hotspots



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw": BASE_DIR+"/data/raw","processed": BASE_DIR+"/data/processed",
        "metadata": BASE_DIR+"/data/metadata","visualizations": BASE_DIR+"/visualizations"}
LAT_MIN,LAT_MAX,LON_MIN,LON_MAX = 5,30,65,95
YEAR_MIN,YEAR_MAX = 2015,2026
import seaborn as sns

In [ ]:
master = pd.read_csv(DIRS['processed']+'/master_table_v1.csv')
print(f'Master table: {master.shape}')
print(f'Ocean zones: {master["Ocean_Zone"].value_counts().to_dict()}')

## Step 1: Identify Hotspot Cells

In [ ]:
# A hotspot = grid cell with Plastic_Density in top 20%
plastic_data = master.dropna(subset=['Plastic_Density_kg_km2'])

threshold = plastic_data['Plastic_Density_kg_km2'].quantile(0.80)
hotspots = plastic_data[plastic_data['Plastic_Density_kg_km2'] >= threshold].copy()

print(f'Hotspot threshold (80th percentile): {threshold:.6f} kg/km²')
print(f'Hotspot cells: {len(hotspots)} / {len(plastic_data)} ({len(hotspots)/len(plastic_data)*100:.1f}%)')
print()
print('Top 10 hotspot cells:')
top10 = hotspots.nlargest(10,'Plastic_Density_kg_km2')[
    ['grid_lat','grid_lon','Ocean_Zone','Plastic_Density_kg_km2','Plastic_Types_Present']
]
print(top10.to_string(index=False))

## Step 2: Heatmap of Plastic Density

In [ ]:
# Create a lat×lon density matrix for heatmap
heat_df = plastic_data.pivot_table(
    index='grid_lat', columns='grid_lon',
    values='Plastic_Density_kg_km2', aggfunc='sum'
)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(heat_df, cmap='YlOrRd', linewidths=0.1, linecolor='gray',
            cbar_kws={'label': 'Plastic Density (kg/km²)'}, ax=ax)
ax.set_title('Plastic Density Hotspot Map — North Indian Ocean\n(Arabian Sea, Bay of Bengal, Andaman Sea)', fontsize=13)
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
plt.tight_layout()
plt.savefig(DIRS['visualizations']+'/Week4_Day1_hotspot_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Hotspot heatmap saved ✅')

In [ ]:
hotspots.to_csv(DIRS['processed']+'/hotspots.csv', index=False)
print(f'Hotspots saved: hotspots.csv ({len(hotspots)} cells) ✅')